# Fase 4 v2 — 06: minimap cenital 'best of the best'

Junta lo bueno: homografía estable por líneas (`VideoHomographyLines`, overlap-gated)
+ **objetos por YOLO** (best.pt: `robot`, `orange_ball`) + porterías por YOLO
(`yellow_zone`/`blue_zone`) para orientar + **minimap con estelas** (`MinimapRenderer`).
SAM3 solo segmenta `green_floor` (para las líneas blancas de la homografía).

Resultado cenital: ~108/110 `fit`, overlay pegado, círculo clavado, 4-6 objetos/frame
en el minimap. Requiere GPU (SAM3 + YOLO).

In [ ]:
import sys, os, subprocess
from pathlib import Path
import numpy as np, cv2, decord
REPO = Path.cwd().resolve()
while not (REPO / 'src/core/field_template.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.core.sam3_loader import load_sam3
from src.core.segmentation import detect_classes_in_frame, _load_classes
from src.core.detectors import detect_boxes
from src.core.minimap_pipeline import _largest_mask, _largest_component, _objects_from_boxes, _box_centroid, _GreedyTracker
from src.core.homography import project_points
from src.core.minimap import MinimapRenderer
from src.core import field_landmarks as fl
from src.core.homography_multifeature import VideoHomographyLines
OUT = REPO / 'notebooks/fase_4_homografia/v2_robust/outputs/videos'; OUT.mkdir(parents=True, exist_ok=True)
bundle = load_sam3(); ALL = _load_classes()
GREEN = [c for c in ALL if c['name']=='green_floor']
YOLO = [c for c in ALL if 'yolo_id' in c]   # robot, orange_ball, yellow_zone, blue_zone
print('YOLO classes:', [c['name'] for c in YOLO])

In [ ]:
TH=np.linspace(0,2*np.pi,60); CIRC=np.stack([fl.CENTER_CIRCLE[0]+fl.CENTER_CIRCLE[2]*np.cos(TH),fl.CENTER_CIRCLE[1]+fl.CENTER_CIRCLE[2]*np.sin(TH)],1)
def c2i(Hi,p): return cv2.perspectiveTransform(np.asarray(p,np.float64).reshape(-1,1,2),Hi).reshape(-1,2)
def overlay_rgb(rgb,H):
    img=rgb.copy()
    if H is None: return img
    Hi=np.linalg.inv(H); rect=[fl.LANDMARK_POINTS[n] for n in ['inner_tl','inner_tr','inner_br','inner_bl']]
    cv2.polylines(img,[c2i(Hi,rect).astype(np.int32)],True,(0,255,0),3,cv2.LINE_AA)
    cl=c2i(Hi,[fl.LANDMARK_POINTS['center_top'],fl.LANDMARK_POINTS['center_bot']]).astype(np.int32); cv2.line(img,tuple(cl[0]),tuple(cl[1]),(255,255,0),3,cv2.LINE_AA)
    cv2.polylines(img,[c2i(Hi,CIRC).astype(np.int32)],True,(255,140,0),3,cv2.LINE_AA); return img

p='/workspace/Meta_Glasses/18abril/Camara_superior/IMG_9933.MOV'; vr=decord.VideoReader(p)
idxs=list(range(0,len(vr),3))[:110]; vh=VideoHomographyLines(); tracker=_GreedyTracker(); renderer=MinimapRenderer()
tmp=str(OUT/'mm_tmp.mp4'); wr=None; ow2=oh2=None
for k,i in enumerate(idxs):
    rgb=vr[i].asnumpy()
    cm=_largest_mask(detect_classes_in_frame(rgb,GREEN,bundle).get('green_floor',[]))
    boxes=detect_boxes(rgb,classes=YOLO,conf=0.25)
    yc=_box_centroid(boxes.get('yellow_zone',[])); bc=_box_centroid(boxes.get('blue_zone',[]))
    if cm is None: H,st,ov=vh.H,'kept',0.0
    else:
        cm=_largest_component(cm); H,st,ov=vh.update(cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR),cm,yc,bc)
    objs=tracker.update(_objects_from_boxes({n:boxes.get(n,[]) for n in ('robot','orange_ball')}))
    projected=[]
    if H is not None and objs:
        feet=np.array([foot for _,_,foot in objs],np.float32); cms=project_points(feet,H)
        for (oid,cls,_),(x,y) in zip(objs,cms): projected.append((oid,cls,float(x),float(y)))
    renderer.update(projected)
    base=overlay_rgb(rgb,H); cv2.putText(base,st+' ov='+format(ov,'.2f')+' obj='+str(len(projected)),(20,46),cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,255,0),3,cv2.LINE_AA)
    comp=cv2.cvtColor(renderer.composite(base),cv2.COLOR_RGB2BGR)
    if wr is None:
        oh,ow=comp.shape[:2]; sc=1100/ow; ow2,oh2=1100,int(oh*sc); wr=cv2.VideoWriter(tmp,cv2.VideoWriter_fourcc(*'mp4v'),12,(ow2,oh2))
    wr.write(cv2.resize(comp,(ow2,oh2)))
wr.release(); fin=str(OUT/'minimap_best_cenital.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',tmp,'-vcodec','libx264','-pix_fmt','yuv420p',fin],check=True); os.remove(tmp)
print('STATS',vh.stats(),'MB',round(os.path.getsize(fin)/1e6,2))

## Notas

- Homografía = `VideoHomographyLines` (líneas blancas vía `green_floor` SAM3 + gate de
  overlap + EMA): estable, no chueca.
- Objetos/porterías = YOLO `best.pt` (la config tiene UNA clase `robot`, no robot_a/b).
- Lateral sigue pendiente (detector de líneas Hough/LSD); aquí el foco es cenital pulido.